# BA 878 Healthcare Analysis

## Data Input

In [1]:
import pandas as pd
import numpy as np

In [2]:
df_raw = pd.read_csv('df_merge.csv')

## Preprocessing

### Drop Columns

In [3]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5659 entries, 0 to 5658
Columns: 153 entries, Unnamed: 0 to Strength
dtypes: bool(1), float64(112), int64(6), object(34)
memory usage: 6.6+ MB


In [4]:
df_raw.dtypes

Unnamed: 0          int64
id                  int64
study_interval      int64
is_weekend           bool
day_in_study        int64
                   ...   
duration          float64
steps_exercise    float64
Cardio            float64
Sports            float64
Strength          float64
Length: 153, dtype: object

In [5]:
df_raw.drop(columns = ['10am_rmssd','10am_lf','10am_hf','12pm_rmssd','12pm_lf','12pm_hf','2pm_rmssd','2pm_lf','2pm_hf','4pm_rmssd','4pm_lf',
                       '4pm_hf','6pm_rmssd','6pm_lf','6pm_hf','8pm_rmssd','8pm_lf','8pm_hf','10pm_rmssd',
                       '10pm_lf','10pm_hf', 'study_interval_x','is_weekend_x','study_interval_y', 'is_weekend_y','study_interval_slp',
                       'is_weekend_slp','study_interval_ct', 'is_weekend_ct','study_interval_hr', 'is_weekend_x_hr','is_weekend_y_hr', 
                       'is_weekend_hr','study_interval_exe','is_weekend_exe'], inplace = True)

In [6]:
df_raw.columns

Index(['Unnamed: 0', 'id', 'study_interval', 'is_weekend', 'day_in_study',
       'phase', 'lh', 'estrogen', 'pdg', 'flow_volume',
       ...
       '8am_hf', 'calories', 'steps', 'averageheartrate', 'calories_exercise',
       'duration', 'steps_exercise', 'Cardio', 'Sports', 'Strength'],
      dtype='object', length=118)

In [7]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5659 entries, 0 to 5658
Columns: 118 entries, Unnamed: 0 to Strength
dtypes: bool(1), float64(85), int64(6), object(26)
memory usage: 5.1+ MB


### Drop Duplicates

In [8]:
df_raw.drop_duplicates()
df_raw.duplicated().sum()

0

### Renaming

In [9]:
df_raw = df_raw.rename(columns={
    'lh': 'hormone_lh',
    'estrogen': 'hormone_estrogen',
    'pdg': 'hormone_pdg'
})

In [10]:
symptom_cols = [
    'flow_volume', 'flow_color', 'appetite', 'exerciselevel', 'headaches',
    'cramps', 'sorebreasts', 'fatigue', 'sleepissue', 'moodswing',
    'stress', 'foodcravings', 'indigestion', 'bloating'
]

df_raw = df_raw.rename(columns={col: f'symptom_{col}' for col in symptom_cols})

In [11]:
glucose_cols = [
    'glucose_12_2_am', 'glucose_2_4_am', 'glucose_4_6_am', 'glucose_6_8_am',
    'glucose_8_10_am', 'glucose_10_12_am', 'glucose_12_2_pm', 'glucose_2_4_pm',
    'glucose_4_6_pm', 'glucose_6_8_pm', 'glucose_8_10_pm', 'glucose_10_12_pm'
]

df_raw = df_raw.rename(columns={
    col: f"glucose_{col.split('_')[1]}{col.split('_')[-1]}" for col in glucose_cols
})

In [12]:
df_raw = df_raw.rename(columns={
    'glucose_min': 'daily_glucose_min',
    'glucose_max': 'daily_glucose_max',
    'glucose_avg': 'daily_glucose_avg'
})


In [13]:
sleep_cols = [
    'composition_score',
    'revitalization_score',
    'duration_score',
    'resting_heart_rate',
    'restlessness'
]

df_raw = df_raw.rename(columns={col: f"sleep_{col}" for col in sleep_cols})


In [14]:
df_raw = df_raw.rename(columns={
    col: f"daily_br_{col.split('_')[0]}_{col.split('_')[-1]}"
    for col in df_raw.columns if 'sleep_breathing_rate' in col
})

In [15]:
df_raw = df_raw.rename(columns={
    'in_default_zone_3': 'time_in_default_zone3',
    'in_default_zone_2': 'time_in_default_zone2',
    'in_default_zone_1': 'time_in_default_zone1',
    'below_default_zone_1':'time_below_default_zone1'
})

In [16]:
bpm_cols = [
    '12am_bpm', '2am_bpm', '4am_bpm', '6am_bpm', '8am_bpm', '10am_bpm',
    '12pm_bpm', '2pm_bpm', '4pm_bpm', '6pm_bpm', '8pm_bpm', '10pm_bpm'
]

df_raw = df_raw.rename(columns={col: f"bpm_{col.split('_')[0]}" for col in bpm_cols})


In [17]:

mapping = {
    'rmssd': 'rmssd',
    'lf': 'low_frequency',
    'hf': 'high_frequency'
}

df_raw = df_raw.rename(columns={
    col: f"{mapping[col.split('_')[1]]}_{col.split('_')[0]}"
    for col in df_raw.columns
    if any(col.endswith(f"_{k}") for k in mapping.keys())
})


### Change Dtypes

In [18]:
df_raw.dtypes

Unnamed: 0          int64
id                  int64
study_interval      int64
is_weekend           bool
day_in_study        int64
                   ...   
duration          float64
steps_exercise    float64
Cardio            float64
Sports            float64
Strength          float64
Length: 118, dtype: object

In [19]:
df_raw['id'] = df_raw['id'].astype('object')

In [20]:
df_raw['timestamp']= pd.to_datetime(df_raw['timestamp'], format='%H:%M:%S').dt.time

In [21]:
df_raw['sleep_start_timestamp']= pd.to_datetime(df_raw['sleep_start_timestamp'], format='%H:%M:%S').dt.time
df_raw['sleep_end_timestamp']= pd.to_datetime(df_raw['sleep_end_timestamp'], format='%H:%M:%S').dt.time
df_raw['sleep_start_timestamp_ct']= pd.to_datetime(df_raw['sleep_start_timestamp_ct'], format='%H:%M:%S').dt.time
df_raw['sleep_end_timestamp_ct']= pd.to_datetime(df_raw['sleep_end_timestamp_ct'], format='%H:%M:%S').dt.time

In [22]:
df_raw.dtypes

Unnamed: 0          int64
id                 object
study_interval      int64
is_weekend           bool
day_in_study        int64
                   ...   
duration          float64
steps_exercise    float64
Cardio            float64
Sports            float64
Strength          float64
Length: 118, dtype: object

In [23]:
type(df_raw['timestamp'].iloc[3])

datetime.time

In [24]:
print(type(df_raw['sleep_start_timestamp'].iloc[3]))
print(type(df_raw['sleep_end_timestamp'].iloc[3]))
print(type(df_raw['sleep_start_timestamp_ct'].iloc[3]))
print(type(df_raw['sleep_end_timestamp_ct'].iloc[3]))

<class 'datetime.time'>
<class 'datetime.time'>
<class 'datetime.time'>
<class 'datetime.time'>


## outliers

In [25]:
import pandas as pd

def detect_outlier_rows(df, name="df"):
    # Select numeric columns only
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
    outlier_mask = pd.Series(False, index=df.index)

    for col in numeric_cols:
        series = df[col].dropna()  # ignore NaN values for stats
        if series.empty:
            continue

        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        # Mark rows that are outliers — skip NaNs
        outlier_mask |= ((df[col] < lower) | (df[col] > upper)) & df[col].notna()

    total_rows = len(df)
    outlier_rows = outlier_mask.sum()
    pct_rows = outlier_rows / total_rows * 100

    print(f" {name} has {outlier_rows:,} rows containing at least one outlier "
          f"(excluding null values), which is {pct_rows:.2f}% of all rows.")
    
    return outlier_rows, pct_rows, outlier_mask

outlier_rows, pct_rows, outlier_mask = detect_outlier_rows(df_raw, "df_raw")



 df_raw has 4,126 rows containing at least one outlier (excluding null values), which is 72.91% of all rows.


In [27]:
import pandas as pd

def detect_glucose_outliers(df):
    glucose_cols = [col for col in df.columns if 'glucose' in col.lower()]
    results = []

    for col in glucose_cols:
        series = df[col].dropna()
        if series.empty:
            continue

        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        # record outlier rows
        outlier_idx = df.index[((df[col] < lower) | (df[col] > upper)) & df[col].notna()]
        results.append({
            'column': col,
            'lower_bound': round(lower, 2),
            'upper_bound': round(upper, 2),
            'outlier_count': len(outlier_idx),
            'outlier_rows': outlier_idx.tolist()
        })

    glucose_outliers = pd.DataFrame(results).sort_values(by='outlier_count', ascending=False).reset_index(drop=True)
    print(" Glucose outlier summary:")
    display(glucose_outliers[['column', 'lower_bound', 'upper_bound', 'outlier_count']])
    return glucose_outliers

glucose_outliers = detect_glucose_outliers(df_raw)


 Glucose outlier summary:


,column,lower_bound,upper_bound,outlier_count
0,glucose_2am,3.60,9.20,199
1,glucose_12pm,3.10,8.70,197
2,glucose_2pm,2.90,9.30,191
3,glucose_4pm,3.00,9.40,191
4,glucose_10am,3.25,8.45,186
5,daily_glucose_max,5.25,12.85,183
6,daily_glucose_avg,4.09,8.31,182
7,glucose_12am,3.18,9.77,179
8,glucose_6am,3.41,8.51,175
9,glucose_4am,3.55,8.75,174


In [29]:
df_raw.head()

,Unnamed: 0,id,study_interval,is_weekend,day_in_study,phase,hormone_lh,hormone_estrogen,hormone_pdg,symptom_flow_volume,...,high_frequency_8am,calories,steps,averageheartrate,calories_exercise,duration,steps_exercise,Cardio,Sports,Strength
0,0,1,2022,True,1,Follicular,2.9,94.2,NaN,Not at all,...,NaN,1542.0,992.0,104.0,61.0,1178000.0,651.0,0.0,1.0,0.0
1,1,1,2022,False,2,Follicular,1.2,226.3,NaN,Not at all,...,NaN,1591.0,838.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,1,2022,False,3,Follicular,3.5,276.8,NaN,Not at all,...,NaN,1755.0,2586.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,1,2022,False,4,Fertility,1.8,322.1,NaN,Not at all,...,NaN,1552.0,1275.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,1,2022,False,5,Fertility,4.6,244.9,NaN,Not at all,...,NaN,1456.0,436.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
df_raw = df_raw.drop(columns=['Unnamed: 0'])

In [35]:
df_raw["study_interval"].value_counts()

study_interval
2022    3698
2024    1961
Name: count, dtype: int64

In [38]:
df_2022 = df_raw[df_raw["study_interval"] == 2022]

In [ ]:
df_2022.info()

In [39]:
for col, n in df_2022.isnull().sum().items():
    print(col, n)


id 0
study_interval 0
is_weekend 0
day_in_study 0
phase 1
hormone_lh 223
hormone_estrogen 224
hormone_pdg 3698
symptom_flow_volume 509
symptom_flow_color 509
symptom_appetite 378
symptom_exerciselevel 378
symptom_headaches 378
symptom_cramps 378
symptom_sorebreasts 378
symptom_fatigue 378
symptom_sleepissue 378
symptom_moodswing 378
symptom_stress 378
symptom_foodcravings 378
symptom_indigestion 378
symptom_bloating 378
glucose_12am 759
glucose_2am 778
glucose_4am 776
glucose_6am 764
glucose_8am 749
glucose_10am 749
glucose_12pm 745
glucose_2pm 729
glucose_4pm 732
glucose_6pm 732
glucose_8pm 727
glucose_10pm 742
daily_glucose_min 590
daily_glucose_max 590
daily_glucose_avg 590
birth_year 0
gender 0
ethnicity 0
education 0
sexually_active 0
self_report_menstrual_health_literacy 90
age_of_first_menarche 0
height2022 1808
weight2022 1538
height2024 2978
weight2024 2708
timestamp 450
sleep_composition_score 450
sleep_revitalization_score 450
sleep_duration_score 450
deep_sleep_in_minutes 4

## Finish and download

In [31]:
df_preprocessed =df_raw.to_csv('df_preprocessed.csv')